Notebook to load and save the predictions

In [1]:
import numpy as np
import pandas as pd 
import mosqlient as mosq 
import matplotlib.pyplot as plt 
from epiweeks import  Week 
from mosqlient.scoring import Scorer
from aux_func import get_data 

In [2]:
import os
from dotenv import load_dotenv

# Access the environment variables
api_key = os.getenv('api_key')

In [3]:
code_to_state = {33: 'RJ', 32: 'ES', 41: 'PR', 23: 'CE', 21: 'MA',
 31: 'MG', 42: 'SC', 26: 'PE', 25: 'PB', 24: 'RN', 22: 'PI', 27: 'AL',
 28: 'SE', 35: 'SP', 43: 'RS', 15: 'PA', 16: 'AP', 14: 'RR',  11: 'RO',
 13: 'AM', 12: 'AC', 51: 'MT', 50: 'MS', 52: 'GO', 17: 'TO', 53: 'DF',
 29: 'BA'}

state_to_code = {value: key for key, value in code_to_state.items()}

geo_dengue = [2931350,2933307,2302503,3119401,
              3549805,3541406,1200401,1200203,
              1716109,4113700,4103701,4104808,
              5201405,5102637,5215231]

geo_chik = [2211001,2931350,3143302,3119401,
            1721000,1716109,4104808,4219507,
            5103403,5102637] 

In [5]:
df_dengue_state = get_data('dengue_state')
df_dengue_city = get_data('dengue_city')
df_chik_state = get_data('chik_state')
df_chik_city = get_data('chik_city')

In [6]:
df_val = pd.read_csv('predictions/validate_models.csv')
df_val = df_val.loc[df_val.dados_corretos == True]
df_val.head()

,model_name,disease,qtd_adm,todos_com_4_validaçoes,challenge,dados_corretos
0,blaiate/3rd_imdc_-unifesp-_-4mosqueteiras-,A90,26,True,dengue_state,True
1,americocunhajr/3rd_imdc_lncc_clidengo26chikung...,A92.0,26,True,chik_state,True
2,americocunhajr/3rd_imdc_lncc_clidengo26dengue,A90,26,True,dengue_state,True
4,DiogoParreira/3rd_imdc_rki_rki_zki_ph,A90,26,True,dengue_state,True
5,DiogoParreira/3rd_imdc_rki_rki_zki_ph_lstm_geo,A90,26,True,dengue_state,True


In [7]:
df_val.challenge.unique()

<ArrowStringArray>
['dengue_state', 'chik_state', 'dengue_city', 'chik_city']
Length: 4, dtype: str

In [8]:
for arrow in df_val.itertuples():

    model_name = arrow.model_name.split('/')[1]

    filename = f'predictions/{arrow.challenge}_{model_name}.csv.gz'

    if os.path.exists(filename):
        pass 

    else: 
        print(model_name)

        df_preds = pd.DataFrame()

        if arrow.challenge.split('_')[1] == 'state': 
            adm_level = 1 
            col = 'adm_1'

            if arrow.challenge.split('_')[0] == 'dengue': 

                df_data = df_dengue_state

            else: 
                df_data = df_chik_state

        elif arrow.challenge.split('_')[1] == 'city':
            adm_level = 2  
            col = 'adm_2'

            if arrow.challenge.split('_')[0] == 'dengue': 

                df_data = df_dengue_city

            else: 
                df_data = df_chik_city


        preds = mosq.get_predictions(api_key=api_key,
                        disease = arrow.disease, 
                        model_name = model_name,
                        adm_level = adm_level)
        

        for pred in preds: 

            if (pred.start == Week(2022, 41).startdate() ) and (pred.end == Week(2023, 40).startdate()):
                val =1
                start = Week(2022, 41).startdate()
                end = Week(2023, 40).startdate()

            elif (pred.start == Week(2023, 41).startdate() ) and (pred.end == Week(2024, 40).startdate()):
                val = 2
                start = Week(2023, 41).startdate()
                end = Week(2024, 40).startdate()

            elif (pred.start == Week(2024, 41).startdate() ) and (pred.end == Week(2025, 40).startdate()):
                val = 3
                start = Week(2024, 41).startdate()
                end = Week(2025, 40).startdate()

            elif (pred.start == Week(2025, 41).startdate() ) and (pred.end == Week(2026, 40).startdate()):
                val = 4
                start = Week(2025, 41).startdate()
                end = Week(2026, 40).startdate()

            else:
                val = None 

            #elif (pred.start == Week(2026, 41).startdate() ) and (pred.end == Week(2027, 40).startdate()):
            #    val = 'Forecast'
            #    start = Week(2026, 41).startdate()
            #    end = Week(2027, 40).startdate()

            if val is not None: 

                df_ = pred.to_dataframe()

                df_[col] = getattr(pred, col)

                df_['id'] = pred.id 

                df_['validation'] = val


                if pred.scores.get("wis") is None: 

                    score = Scorer(
                        api_key=api_key,
                        df_true=df_data.loc[df_data[col] == getattr(pred, col)],
                        pred=df_,
                    )

                    df_['wis'] = score.wis[1]['pred']

                else: 
                    df_['wis'] = pred.scores.get("wis")


                df_preds = pd.concat([df_preds, df_], ignore_index=True)


        df_preds.to_csv(filename, index = False)


3rd_imdc_isi_isi-dengue
